In [30]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
%autoreload
import sys
import copy
import os 

sys.path.append('../../style_generation_pipeline/')
os.environ['LORA_BASEMODEL_CHECKPOINT_PATH'] = "/mnt/swordfish-pool2/milad/hiatus-data/models/original_ta2_performers_data_model_50k_hrs_continued_final_model/" 

import generate_explanations
from utils import hiatus_utils, explanation_interfaces

In [32]:
import json
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from sklearn.preprocessing import minmax_scale
import random
from data import get_aa_data_from_original_format

pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [33]:
def generate_hiatus_explanations(data_point, interp_space_path, interp_space_rep_path, style_feat_clm, style_feat_summary_clm, explanation_interf, cluster_lvl=True, random_feature_assignment=False, max_author_docs=3):

    query_authors, candidate_authors, queries_df, candidates_df, ground_truth_assignment = data_point[0], data_point[1], data_point[2], data_point[3], data_point[4] 
    
    output_json = []
    for q_author_id in query_authors:
    
        q_author_documents = queries_df[queries_df.authorID == q_author_id]['fullText'].tolist()[0]
        c_author_documents = candidates_df.fullText.tolist()

        result = generate_explanations.explain_model_prediction_over_author(model_path, interp_space_path, interp_space_rep_path, \
                                                                            q_author_documents, c_author_documents, top_c=top_c, style_feat_clm=style_feat_clm, \
                                                                            style_feat_summary_clm=style_feat_summary_clm, cluster_lvl=cluster_lvl)

        latent_rank = result[0]
        interp_rank = result[1]

        if cluster_lvl == True:
            style_reps = result[2]
            style_reps_summ = result[3]
            author_sim_to_styles = result[4]
            query_author_rep_doc_id = result[5]
            candidate_authors_rep_doc_ids = result[6]
        else:
            query_author_style_feats = result[2]
            candidate_authors_style_feats = result[3]
            selected_feats = result[4]
            query_author_rep_doc_id = result[5]
            candidate_authors_rep_doc_ids = result[6]

        # print(q_author_id)
        # print(latent_rank)
        # print(interp_rank)
        
        instance_json = {
            "Q_fullText": "\n\n".join(["Document {}: \n{}".format(i+1,d) for i, d in enumerate(q_author_documents[:max_author_docs])]),
            #"Q_fullText": q_author_documents[query_author_rep_doc_id],
            "Q_authorID": q_author_id,
            "latent_rank" : latent_rank.tolist(),
            "system_rank" : [{"id": x+1, "title": 'Author {}'.format(x+1), "body": ""} for i, x in enumerate(latent_rank.tolist())],
            "rank_1": "1. Author {}".format(latent_rank[0]+1),
            "rank_2": "2. Author {}".format(latent_rank[1]+1),
            "rank_3": "3. Author {}".format(latent_rank[2]+1),
            "interp_rank" : interp_rank.tolist(),
        }

        #print(instance_json['system_rank'])
        #Find candidate authors order
        candidate_author_order = []
        for order, c_author_id in enumerate(candidate_authors):
            instance_json['a{}_fullText'.format(order)] = "\n\n".join(["Document {}: \n{}".format(i+1,d) for i, d in enumerate(c_author_documents[order][:max_author_docs])])
            #instance_json['a{}_fullText'.format(order)] = c_author_documents[order][candidate_authors_rep_doc_ids[order]]
            instance_json['a{}_authorID'.format(order)] = c_author_id
            order+=1
            candidate_author_order.append(candidate_authors.index(c_author_id))

        if cluster_lvl:
            instance_json['explanation'] = hiatus_utils.build_explanation_interface1(explanation_interf, author_sim_to_styles, style_reps_summ, candidate_authors, random_feature_assignment=random_feature_assignment)
        else:
            instance_json['explanation'] = hiatus_utils.build_explanation_interface2(explanation_interf, query_author_style_feats, candidate_authors_style_feats, selected_feats, random_feature_assignment=random_feature_assignment)

        #Find the ground-truth labels (who wrote the query document)
        query_author_idx = query_authors.index(q_author_id)
        candidate_atuthor_labels = [ground_truth_assignment[query_author_idx][a_idx] for a_idx in candidate_author_order]
        gt_idx = candidate_atuthor_labels.index(1)
        instance_json["gt_idx"] = gt_idx
        
        output_json.append(instance_json)

    return output_json[0] # it is always one query author per instance

In [34]:
def run_explanation_experiment(input_path, model_path, interp_space_path, interp_space_rep_path, data_generator, output_path, top_c=3, top_k=10):
    output = []
    idx=0

    latent_pred_accuracy = []
    interp_pred_accuracy = []
    for data_point in data_generator:

        # Generate explanations on feature-level
        instance_exp = generate_hiatus_explanations(data_point, interp_space_path, interp_space_rep_path, 'llm_tfidf_weights', 'llm_tfidf_weights', explanation_interfaces.exp_interface_2, cluster_lvl=False)
        instance_exp['source'] = 'feature_lvl_explanation_tfidf'

        latent_pred_accuracy.append(1 if instance_exp['gt_idx'] == instance_exp['latent_rank'][0] else 0)
        interp_pred_accuracy.append(1 if instance_exp['gt_idx'] == instance_exp['interp_rank'][0] else 0)
        
        # Generate random equivelant explanations on feature-level
        instance_exp_random = generate_hiatus_explanations(data_point, interp_space_path, interp_space_rep_path, 'llm_tfidf_weights', 'llm_tfidf_weights', explanation_interfaces.exp_interface_2, cluster_lvl=False, random_feature_assignment=True)
        instance_exp_random['source'] = 'feature_lvl_explanation_tfidf_random'
        
        # Create an instance with no explanation
        instance_wo_exp = {x[0]: x[1] for x in instance_exp.items()}
        instance_wo_exp['explanation'] = '<p>No Explanation</p>'
        instance_wo_exp['source'] = 'no_explanation'
        
        output.append(instance_wo_exp)
        output.append(instance_exp)
        output.append(instance_exp_random)
    
        # generate explanations on cluster level
        for feat_clm in ['llm_tfidf_rep', 'g2v_tfidf_rep']:
            instance_exp = generate_hiatus_explanations(data_point, interp_space_path, interp_space_rep_path, feat_clm, feat_clm, explanation_interfaces.exp_interface_1_1)
            instance_exp['source'] = feat_clm
            output.append(instance_exp)
            
            instance_exp_random = generate_hiatus_explanations(data_point, interp_space_path, interp_space_rep_path, feat_clm, feat_clm, explanation_interfaces.exp_interface_1_1, random_feature_assignment=True)
            instance_exp_random['source'] = feat_clm + '_random'
            output.append(instance_exp_random)
        
        #open(data_point + '/explanation.html', 'w').write(instance_exp['explanation'])
        idx+=1

    print('Latent Accuracy ', sum(latent_pred_accuracy)/len(latent_pred_accuracy))
    print('Interp Accuracy ', sum(interp_pred_accuracy)/len(interp_pred_accuracy))
    json.dump(output, open(output_path, 'w'))

## Experiment to generate explanation on IARPA's pilot data

In [12]:
#model_path = 'aa_model-luar'
model_path = 'luar-mud'
input_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability-pilot-samples/'
top_c=3

interp_space_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space.pkl'
interp_space_rep_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space_representations.json'

run_explanation_experiment(input_path, model_path, interp_space_path, interp_space_rep_path, hiatus_utils.get_iarapa_pilot_data(input_path),
                           '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/iarapa_pilot_explanations.json', top_c=3, top_k=10)

## Experiment on Generating explanations on samples from HRS

In [35]:
#model_path = 'aa_model-luar'
model_path = 'luar-mud'
input_path = '/mnt/swordfish-pool2/milad/hiatus-data/phase_2/internal_cross_fold_cross_genre/fold_1/test/TA2/hrs2_09-24-24_english_crossGenre-combined/'
top_c=3

interp_space_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space.pkl'
interp_space_rep_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/lora_ta2_interp_space_clusters/interpretable_space_representations.json'

run_explanation_experiment(input_path, model_path, interp_space_path, interp_space_rep_path, hiatus_utils.get_hrs_data(input_path, random_seed=737),
                           '/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/hrs_explanations.json', top_c=3, top_k=10)

Latent Accuracy  1.0
Interp Accuracy  0.8


In [27]:
feature_weights = [np.log(x) for x in [[1,10,100]]]
feature_weights_color = [hiatus_utils.get_color_gradient(x) for x in feature_weights]

In [28]:
feature_weights_color

[['#90ee90', '#489058', '#013220']]

In [29]:
feature_weights

[array([0.        , 2.30258509, 4.60517019])]

## Analyze documents style description

In [8]:
interp_space_path = '/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/interpretable_space.pkl'
model_path = 'aa_model-luar'
explanation_interf = explanation_interfaces.exp_interface_1
input_path = '/mnt/swordfish-pool2/milad/hiatus-data/explainability-pilot-samples/'
top_c=3

In [9]:
path_to_input = '/mnt/swordfish-pool2/milad/hiatus-data/explainability-pilot-samples/samples_26_query_79_first/'

In [10]:
candidates_file = list(glob.glob(path_to_input + '/data/*_candidates.jsonl'))[0]
queries_file    = list(glob.glob(path_to_input + '/data/*_queries.jsonl'))[0]
grount_truth_file = list(glob.glob(path_to_input + '/groundtruth/*_groundtruth.npy'))[0]
q_labels_file = glob.glob(path_to_input + '/groundtruth/*_query-labels.txt')[0]
c_labels_file = glob.glob(path_to_input + '/groundtruth/*_candidate-labels.txt')[0]

candidates_df = pd.read_json(candidates_file, lines=True)
queries_df = pd.read_json(queries_file, lines=True)

queries_df['authorID'] = queries_df.authorIDs.apply(lambda x: x[0])
candidates_df['authorID'] = candidates_df.authorSetIDs.apply(lambda x: x[0])

queries_df = queries_df.groupby('authorID').agg({'fullText': lambda x: list(x)}).reset_index()
candidates_df = candidates_df.groupby('authorID').agg({'fullText': lambda x: list(x)}).reset_index()
    
ground_truth_assignment = np.load(open(grount_truth_file, 'rb'))
candidate_authors = [a[2:-3] for a in  open(c_labels_file).read().split('\n')][:-1]
query_authors = [a[2:-3] for a in  open(q_labels_file).read().split('\n')][:-1]

#print(ground_truth_assignment)
#print(candidate_authors)
#print(query_authors)

In [11]:
query_authors

['9265b1f9-31f5-58e5-bfb7-c5979de68d47']

In [12]:
candidate_authors

['07a9a4f0-faed-9b92-fd68-8f47cdc969d5',
 'a83b5d4d-0e79-7ee6-f61b-29b0b059f220',
 'd53e9f35-4c07-3b87-43fc-d52e5f127c6c']

In [13]:
candidate_authors[-1]

'd53e9f35-4c07-3b87-43fc-d52e5f127c6c'

In [14]:
ground_truth_assignment

array([[1, 0, 0]])

In [15]:
matched_candidate_docs = candidates_df[candidates_df.authorID == '07a9a4f0-faed-9b92-fd68-8f47cdc969d5'].fullText.tolist()[0]
other_candidate_docs = candidates_df[candidates_df.authorID != '07a9a4f0-faed-9b92-fd68-8f47cdc969d5'].fullText.tolist()[0] #just pick one other candidate author
query_documents = queries_df.fullText.tolist()[0]

In [16]:
matched_candidate_docs

['The elves had a very distinguished and strict military system, and it included a complex training system that ensured the continued supply of experienced fighters to the army, and this system was based on the soldiers who survived the battles training the new soldiers, as the military system stipulated that the fighters who spent more than 30 years in the war should go to their homeland to train new soldiers in the use of weapons, where the soldier had to prove his competence and strength in order to have the honor of training soldiers.\n\n  But because of the many wars caused by <PERSON> in the first age, and the wars caused by <PERSON> in the second and third ages, where the battles lasted for months, and the siege of castles lasted for years, the elves suffered great losses in their lives, but these losses It was concentrated in the ranks of the swords and spearmen who were always forced to physically interact with the enemy forces, which exposed them to great and permanent danger

In [17]:
other_candidate_docs

["Burning wood has been a local source of warmth for ages, however, while it may provide warmth, it is important to know that burning wood can produce harmful gases such as benzene, and other gases such as carbon monoxide and nitrogen oxides. When organic matter burns, large polycyclic aromatic compounds are released due to combustion. This occurs not only with wood, but also with any other organic matter. Understanding why benzene form in a wood fire, will require us to know what wood is, and how it burns.\n\n  Wood is made up of biomass (matter composed primarily of carbon, hydrogen, and oxygen). This includes things like dry grass, leaves, fibers, animal carcasses, sugar, fat, and coal etc. All of these things contain cellulose or similar molecules, which consists of long chains of aromatic rings.\n\n  When wood is heated up, the molecules composing it start to break apart, in the presence of oxygen, the wood will start to oxidize, this process is what we know as combustion. If ther

In [18]:
query_documents

['The Sahara Desert extends over an area of 9,200,000 square kilometers and is the largest desert in the world. It is one of the driest regions in the world with an average of 25 mm of rain per year. In the past, it was a green area full of rivers, 3 million years ago, until climate change, and the occurrence of several major earthquakes led to the formation of new mountains that blocked the moisture of the sea, and changed the course of the rivers in them, all of this led to the cutting off of water resources and turning it into a dry area with high temperatures, unfit for most types of life.\n\n  Climate change is expected to continue to cause bad consequences for the Sahara desert, as studies indicate that rainfall is expected to decrease by 20 to 30 percent in the next 40 years, and this will lead to more desertification and the destruction of more green spaces on The borders of deserts in general, and will lead to an increase in the borders of the Sahara desert by no less than 7 p

In [19]:
q_documents_reps , q_documents_clusters = generate_explanations.get_documents_style_descriptions(query_documents, model_path, interp_space_path, top_c=3, top_k=5)

TypeError: get_documents_style_descriptions() missing 2 required positional arguments: 'interp_space_rep_path' and 'style_feat_clm'

In [ ]:
c_documents_reps , c_documents_clusters = generate_explanations.get_documents_style_descriptions(matched_candidate_docs, model_path, interp_space_path, top_c=3, top_k=5)

In [ ]:
o_documents_reps , o_documents_clusters = generate_explanations.get_documents_style_descriptions(other_candidate_docs, model_path, interp_space_path, top_c=3, top_k=5)

In [ ]:
q_documents_clusters

In [ ]:
c_documents_clusters

In [ ]:
o_documents_clusters

In [ ]:
q_documents_reps

In [ ]:
c_documents_reps

In [ ]:
o_documents_reps

In [ ]:
interpretable_space = pkl.load(open(interp_space_path, 'rb'))
    
del interpretable_space[-1] #DBSCAN generate a cluster -1 of all outliers. We don't want this cluster
print("# clusters:", len(interpretable_space))
dimension_to_latent = {key: interpretable_space[key][0] for key in interpretable_space}
dimension_to_style  = {key: interpretable_space[key][1] for key in interpretable_space}